# AgriVision: Plant Disease Detection Training Notebook

This notebook demonstrates how to train a **ResNet50** model for plant disease classification using the **PlantVillage** dataset. This is the process used to create models like `mesabo/agri-plant-disease-resnet50`.

In [ ]:
# 1. Install Dependencies
!pip install torch torchvision transformers safetensors pillow tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from transformers import ResNetForImageClassification, AutoConfig
from safetensors.torch import save_file
import os
from tqdm import tqdm

## 2. Data Preparation

We use standard ResNet preprocessing: resize to 256, center crop to 224, and normalize with ImageNet stats.

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Assuming your dataset is in a folder named 'data/PlantVillage'
# data_dir = 'data/PlantVillage'
# image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
# dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=True) for x in ['train', 'val']}

## 3. Model Definition

We use a pre-trained ResNet50 and modify the final layer for the number of classes in the dataset (typically 38 for PlantVillage).

In [ ]:
num_classes = 38
model = models.resnet50(pretrained=True)

# Freeze layers if you want to do transfer learning
# for param in model.parameters():
#     param.requires_grad = False

# Replace final layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

## 4. Training Loop

Standard training loop with CrossEntropyLoss and Adam optimizer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        model.train()
        
        # Loop over training data
        # ... (Implementation details)
        print("Training in progress...")
        
    return model

## 5. Exporting to Safetensors

Once the model is trained, we can save it in the `safetensors` format for use in the web app.

In [ ]:
# Save the weights
tensors = model.state_dict()
save_file(tensors, "model.safetensors")

print("Model saved as model.safetensors")

## 6. Creating config.json

To use with the `transformers` library, you also need a `config.json` mapping your indices to labels.

In [ ]:
import json

# Example label mapping
id2label = {"0": "Apple___Apple_scab", "1": "Apple___Black_rot", "#": "..."}

config = {
    "architectures": ["ResNetForImageClassification"],
    "model_type": "resnet",
    "num_labels": 38,
    "id2label": id2label,
    "label2id": {v: k for k, v in id2label.items()}
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("config.json created")